# Module 04 — Lab: Multimodal

Generates a placeholder chart PNG and a tiny PDF, then sends them to Claude.
Replace with your own files for more interesting outputs.

In [ ]:
import os, io, base64, pathlib
from dotenv import load_dotenv
from anthropic import Anthropic
from PIL import Image, ImageDraw, ImageFont

load_dotenv('../.env')
client = Anthropic()
MODEL = os.getenv('ANTHROPIC_MODEL', 'claude-sonnet-4-6')
ART = pathlib.Path('artifacts'); ART.mkdir(exist_ok=True)

## 1. Generate a simple bar chart image

In [ ]:
img = Image.new('RGB', (640, 360), 'white')
d = ImageDraw.Draw(img)
bars = [('Mon', 42), ('Tue', 67), ('Wed', 55), ('Thu', 91), ('Fri', 73)]
x = 60
for label, val in bars:
    h = val * 3
    d.rectangle([x, 300-h, x+60, 300], fill='steelblue')
    d.text((x+18, 305), label, fill='black')
    d.text((x+18, 300-h-15), str(val), fill='black')
    x += 100
d.text((220, 20), 'Weekly Active Users (k)', fill='black')
png_path = ART / 'chart.png'
img.save(png_path)
print('saved', png_path)

## 2. Send the image as base64 inline

In [ ]:
b64 = base64.standard_b64encode(png_path.read_bytes()).decode()

r = client.messages.create(
    model=MODEL, max_tokens=512,
    messages=[{
        'role':'user',
        'content':[
            {'type':'text','text':'Extract the chart values as JSON: {"day": value, ...}. Reply with JSON only.'},
            {'type':'image','source':{'type':'base64','media_type':'image/png','data': b64}},
        ],
    }],
)
print(r.content[0].text)
print(f'\ntokens: in={r.usage.input_tokens} out={r.usage.output_tokens}')

## 3. Upload a PDF to the Files API (beta)

We'll build a tiny PDF with a known fact, upload it, ask about it, and check citations.

In [ ]:
# Build a one-page PDF with pypdf-friendly content using reportlab if available,
# else fall back to a plain-text doc upload.
pdf_path = ART / 'mini.pdf'
try:
    from reportlab.pdfgen import canvas
    c = canvas.Canvas(str(pdf_path))
    c.drawString(100, 750, 'Annual Report 2026')
    c.drawString(100, 720, 'Revenue grew 27% year over year to $48.2M.')
    c.drawString(100, 700, 'Headcount: 142 (up from 118).')
    c.save()
    print('built PDF with reportlab')
except ImportError:
    pdf_path.write_text('Annual Report 2026\nRevenue grew 27% year over year to $48.2M.\nHeadcount: 142.')
    print('reportlab not installed; wrote .txt as fallback. pip install reportlab to test PDF path.')

In [ ]:
# Files API call — uses the beta namespace at the time of writing.
# If your SDK version exposes it differently, see https://docs.claude.com/en/docs/build-with-claude/files
try:
    uploaded = client.beta.files.upload(file=open(pdf_path, 'rb'))
    print('uploaded:', uploaded.id)
    doc_block = {
        'type':'document',
        'source':{'type':'file','file_id': uploaded.id},
        'title':'Annual Report 2026',
        'citations':{'enabled': True},
    }
except Exception as e:
    print(f'Files API unavailable ({e}); falling back to base64 document block.')
    pdf_b64 = base64.standard_b64encode(pdf_path.read_bytes()).decode()
    doc_block = {
        'type':'document',
        'source':{'type':'base64','media_type':'application/pdf','data': pdf_b64},
        'title':'Annual Report 2026',
        'citations':{'enabled': True},
    }

In [ ]:
r = client.messages.create(
    model=MODEL, max_tokens=512,
    messages=[{
        'role':'user',
        'content':[doc_block, {'type':'text','text':'What was revenue growth? Cite the report.'}],
    }],
)
for b in r.content:
    if b.type == 'text':
        print(b.text)
        for c in (getattr(b, 'citations', None) or []):
            print('   cite:', getattr(c, 'cited_text', c))

## 4. Image token estimator

In [ ]:
def image_token_estimate(w, h):
    return round((w * h) / 750)

for size in [(512,512), (1024,1024), (2048,2048), (4096,4096)]:
    print(size, '~', image_token_estimate(*size), 'tokens')